# SpecTNT — End-to-End Colab Training

Trains all 3 model configs (HarmonicCNN, SpecTNT, SpecTNT+CTL) across 4 folds.

**Pipeline:** Clone repos → Download audio → Precompute features → Train → Show results

In [ ]:
import os, sys, json, time, csv, glob, subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import pandas as pd

print("Environment ready")

## 1. Clone Repositories & Install Dependencies

In [ ]:
# Clone the main repo
REPO = "https://github.com/fharookshaik/BTU_RM_So26.git"
if not os.path.isdir("BTU_RM_So26"):
    !git clone {REPO}
%cd BTU_RM_So26

# Install deps
!pip install -q . 2>&1 | tail -5

# Clone the Harmonix dataset (annotations + metadata)
harmonix_dir = "data/harmonixset"
os.makedirs("data", exist_ok=True)
if not os.path.isdir(harmonix_dir):
    !git clone https://github.com/urinieto/harmonixset.git "{harmonix_dir}"

print(f"\nHarmonix annotations: {len(os.listdir(os.path.join(harmonix_dir, 'dataset', 'segments')))} segment files")

## 2. Download Audio (Parallel + Resumable)

Downloads `.wav` files from YouTube URLs. **Skips existing files** so re-running resumes where it left off.
Failsafe: failed URLs are saved to `download_failed.json` for retry.

In [ ]:
AUDIO_DIR = os.path.join(harmonix_dir, "audio")
URL_CSV   = os.path.join(harmonix_dir, "dataset", "youtube_urls.csv")
FAILED_LOG = "download_failed.json"

os.makedirs(AUDIO_DIR, exist_ok=True)

# Load existing files
existing = {os.path.splitext(os.path.basename(p))[0] for p in glob.glob(os.path.join(AUDIO_DIR, "*.wav"))}
print(f"Already have {len(existing)} .wav files")

# Load previous failures if any
failed_already = set()
if os.path.exists(FAILED_LOG):
    failed_already = set(json.load(open(FAILED_LOG)))
    print(f"Previously failed: {len(failed_already)}")

# Read URL list
entries = []
with open(URL_CSV) as f:
    for row in csv.DictReader(f):
        fid = row["File"]
        if fid not in existing and fid not in failed_already:
            entries.append(row)

total_to_download = len(entries)
print(f"Need to download: {total_to_download}")

if total_to_download == 0:
    print("All files already downloaded!")
else:
    def download_one(row):
        fid, url = row["File"], row["URL"]
        out_path = os.path.join(AUDIO_DIR, f"{fid}.%(ext)s")
        try:
            subprocess.run([
                "yt-dlp", "-x", "--audio-format", "wav",
                "--audio-quality", "0",
                "--postprocessor-args", "ffmpeg:-ac 1 -ar 16000",
                "-o", out_path,
                "--no-playlist", "--quiet", "--no-warnings",
                url,
            ], check=True, capture_output=True, timeout=120)
            return (fid, True, None)
        except Exception as e:
            return (fid, False, str(e))

    failed = []
    total_all = len(existing) + len(failed_already) + len(entries)
    done_count = len(existing) + len(failed_already)
    with ThreadPoolExecutor(max_workers=4) as pool:
        futures = {pool.submit(download_one, e): e for e in entries}
        for fut in as_completed(futures):
            fid, ok, err = fut.result()
            done_count += 1
            if ok:
                print(f"[{done_count}/{total_all}] {fid} OK")
            else:
                print(f"[{done_count}/{total_all}] {fid} FAILED: {err[:60]}")
                failed.append(fid)

    json.dump(failed, open(FAILED_LOG, "w"))
    print(f"\nDone. Failed: {len(failed)}. Re-run this cell to retry.")

In [ ]:
# Quick sanity: available / total audio files
audio_count = len(glob.glob(os.path.join(AUDIO_DIR, "*.wav")))
yt_count = sum(1 for _ in open(URL_CSV)) - 1  # -1 for header
print(f"Audio files: {audio_count} / {yt_count} total tracks")
if audio_count < yt_count * 0.5:
    print("WARNING: Less than 50% audio coverage. Training on available files only.")

## 3. Precompute Features

Computes mel-spectrogram features (`.npy`) from downloaded audio. Skips existing features.

In [ ]:
from src.features import precompute_features, load_features
from src.data_utils import get_songs_with_audio

songs = get_songs_with_audio()
print(f"Songs with audio: {len(songs)}")

paths = precompute_features(songs, force=False)
feature_count = sum(1 for p in paths.values() if os.path.exists(p))
print(f"Features ready: {feature_count} / {len(songs)}")

if feature_count < len(songs):
    # Force compute any missing features from audio
    paths = precompute_features(songs, force=True)
    feature_count = sum(1 for p in paths.values() if os.path.exists(p))
    print(f"Features after force: {feature_count} / {len(songs)}")

## 4. Train All Experiments

Runs 3 models × 4 folds = 12 training sessions with per-run metrics.

In [ ]:
import subprocess, sys, time

models = ["harmonic_cnn", "spectnt", "spectnt_ctl"]
folds  = list(range(4))

for model in models:
    for fold in folds:
        print(f"\n{'='*60}\nStarting: {model}, Fold {fold}\n{'='*60}")
        cmd = [sys.executable, "main.py", "--model", model, "--fold", str(fold)]
        t0 = time.time()
        result = subprocess.run(cmd, capture_output=True, text=True)
        elapsed = time.time() - t0
        print(result.stdout[-1000:] if len(result.stdout) > 1000 else result.stdout)
        if result.returncode != 0:
            print(f"ERROR (rc={result.returncode}):", result.stderr[-500:])
        print(f"Completed in {elapsed:.1f}s")

## 5. Results Summary

Loads `outputs/results/results.json` and displays per-model, per-fold metrics as tables.

In [ ]:
results_path = "outputs/results/results.json"
if not os.path.exists(results_path):
    print("No results found yet. Complete Step 4 first.")
else:
    all_results = json.load(open(results_path))
    for model_name, data in all_results.items():
        print(f"\n{'='*60}")
        print(f"{model_name}")
        print(f"{'='*60}")
        df = pd.DataFrame(data["per_fold"])
        df.index = [f"Fold {i}" for i in range(len(df))]
        print(df.to_string())
        print(f"\nAverage: {data['average']}")